In [29]:
import networkx as nx
from bp.world import random_grid_world, Scenario

In [30]:
instrumented_fraction = 0 # .5

demands = {
    # ((0, 0), (3, 3)): 10,
    # ((1, 0), (1, 3)): 5,
    ((1, 0), (1, 3)): 15,
}

world = random_grid_world(
    rows=4,
    cols=4,
    demands=demands,
    seed=0,
    instrumented_fraction=instrumented_fraction,
)
G = world.network.graph
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")
print(f"{len(world.I) / len(world.A) * 100:.2f}% instrumented (target={instrumented_fraction * 100:.2f}%)")

world.total_population=15
world.total_demand((0, 0), (3, 3))=0
world.total_demand((1, 0), (1, 3))=15
0.00% instrumented (target=0.00%)


In [31]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("asymmetric bp")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

In [32]:
accident_congestion = 10
target_edge = ((1, 0), (1, 1))

# NOTE: disregarding instrumentation for now
# G.edges[target_edge]["instrumented"]
# uninstrumented_edge = next(iter(A))
# instrumented_edge = next(iter(I))

G.edges[target_edge]["travel_time"] += accident_congestion
travel_time = dict(nominal.travel_time)
travel_time[target_edge] += accident_congestion
accident = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal.discomfort,
    hazard=nominal.hazard,
    cost=nominal.cost,
    emissions=nominal.emissions,
    policing=nominal.policing
)

# (mu: scenario, prior: prob)
scenarios = {"nominal": (nominal, .8), "accident": (accident, .2)}

assert sum(mu for (_, mu) in scenarios.values()) == 1, "priors must represent valid probabilities"

In [33]:
from itertools import pairwise, islice
from collections.abc import Sequence

from bp.world import Arc, Node

def edge_path(path: Sequence[Arc]) -> Sequence[Node]:
    """Given a path consisting of vertices, return the corresponding edge-path"""
    return list(pairwise(path))

# k shortest routes
k = 2
R = {}

for plr in world.individuals:
    origin, dest = plr.demand.origin, plr.demand.destination
    od = (origin, dest)
    if od not in R:
        paths = nx.shortest_simple_paths(G, origin, dest, weight="travel_time")
        R[od] = [edge_path(path) for path in islice(paths, k)]

print(f"k=")
assert len(R) == len(demands), "# od pairs != # demands"

k=


In [ ]:
# original: tau[a, x] := average cost for (x + 1) players on arc a
# modified: tau[omega, a, x] := average cost for x players on arc a under scenario omega
tau = {}
for scenario_name, (omega, _) in scenarios.items():
    for a in world.A:
        for k in range(n + 1):
            tau[scenario_name, a, k] = omega.travel_time[a] * (1 + alpha * ((k - 1) / c[a]) ** beta)

from itertools import chain

# optimization bc sparsely using arcs
# double flatten routes into generator over all arcs
active_arcs = set(chain.from_iterable(chain.from_iterable(R.values())))

In [35]:
r = {} # r[omega, od, r_i] := total flow on route r_i for OD pair in state omega
for scenario_name in scenarios:
    for od, paths in R.items():
        demand_od = world.total_demand(od[0], od[1])
        for r_i in range(len(paths)):
            r[scenario_name, od, r_i] = model.addVar(
                vtype=GRB.INTEGER, lb=0, ub=demand_od, name=f"r_{scenario_name}_{od}_{r_i}"
            )

x = {} # x[omega, a] := flow on arc a
cost_arc = {} # cost_arc[omega, a] = realized travel time on arc a
for scenario_name in scenarios:
    for a in active_arcs:
        x[scenario_name, a] = model.addVar(
            vtype=GRB.INTEGER, lb=0, ub=n, name=f"x_{scenario_name}_{a}"
        )
        cost_arc[scenario_name, a] = model.addVar(lb=0, name=f"cost_{scenario_name}_{a}")

# constraint: route flow
for scenario_name in scenarios:
    # \sum_{r_i \in k} r[\omega, od, r_i] = demand for od
    for od, routes in R.items():
        demand_od = world.total_demand(od[0], od[1])
        model.addConstr(
            gp.quicksum(r[scenario_name, od, r_i] for r_i in range(len(routes))) == demand_od,
            name=f"demand_{scenario_name}_{od}"
        )

    # the induced arc flows
    for a in active_arcs:
        arc_flow = gp.quicksum(
            r[scenario_name, od, r_i]
            for od, routes in R.items()
            for r_i, route in enumerate(routes) if a in route
        )
        model.addConstr(x[scenario_name, a] == arc_flow, name=f"arc_vol_{scenario_name}_{a}")

# constraint
for scenario_name in scenarios:
    for a in active_arcs:
        model.addGenConstrPWL(
            x[scenario_name, a],
            cost_arc[scenario_name, a],
            list(range(n + 1)),
            [tau[scenario_name, a, k] for k in range(n + 1)],
            name=f"constr_cost_{scenario_name}_{a}"
        )

for od, routes in R.items():
    for r_i, route in enumerate(routes):
        for r_dev, route_dev in enumerate(routes):
            if r_i != r_dev:
                obedience_expr = 0
                for scenario_name, (omega, mu) in scenarios.items():
                    cost_rec = gp.quicksum(cost_arc[scenario_name, a] for a in route)
                    cost_dev = gp.quicksum(cost_arc[scenario_name, a] for a in route_dev)
                    obedience_expr += mu * r[scenario_name, od, r_i] * (cost_rec - cost_dev)

                model.addConstr(obedience_expr <= 0, name=f"obed_{od}_{r_i}_to_{r_dev}")

# objective
for scenario_name, (_, mu) in scenarios.items():
    for a in active_arcs:
        model.setPWLObj(
            x[scenario_name, a],
            list(range(n + 1)),
            [mu * k * tau[scenario_name, a, k] for k in range(n + 1)]
        )

model.setParam("NonConvex", 2)
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"
print(f"{model.NumConstrs=}")
print(f"{model.NumVars=}")

print(f"Expected Cost: {model.ObjVal:.2f}")
for scenario_name in scenarios:
    print(f"\nState: {scenario_name}")
    for od, paths in R.items():
        for r_i in range(len(paths)):
            flow = int(r[scenario_name, od, r_i].X)
            if flow != 0:
                print(f"Recommend Route {r_i} for {od}: {flow} users")

AssertionError: Optimization failed: 3